# StorkeyHopfield vs MLP — Depth, Few-Shot, and Noise-Robustness Demo

Demonstrates that StorkeyHopfield behaves as a composable graph element in
FabricPC: substituting one or more Linear hidden layers with StorkeyHopfield
should produce accuracy gains that accumulate with the *number* of
substitutions, particularly when the input is noisy and the training set
has enough samples for the attractors to form usable class prototypes.

**Architecture (same depth-4 graph across all arms, hidden width = 64):**

```
"MLP"             pixels(784) -> 64(Linear) -> 64(Linear)          -> 64(Linear)          -> 10(softmax+CE)
"1hopfield"       pixels(784) -> 64(Linear) -> 64(StorkeyHopfield) -> 64(Linear)          -> 10(softmax+CE)
"1hopfield-late"  pixels(784) -> 64(Linear) -> 64(Linear)          -> 64(StorkeyHopfield) -> 10(softmax+CE)
"2hopfield"       pixels(784) -> 64(Linear) -> 64(StorkeyHopfield) -> 64(StorkeyHopfield) -> 10(softmax+CE)
```

**Quick run results** (default args, RTX3090, cuda13, jax 0.10.2):

```
Cell summary:
  MLP            : 58.32%
  1hopfield      : 62.86%
  1hopfield-late : 61.68%
  2hopfield      : 64.26%
```

## Imports & Setup

In [1]:
from typing import Dict, List, Sequence, Tuple

import numpy as np
import jax
import optax

from fabricpc.nodes import Linear, IdentityNode, NodeBase, StorkeyHopfield
from fabricpc.core.topology import Edge
from fabricpc.graph_assembly import TaskMap, graph
from fabricpc.graph_initialization import initialize_params
from fabricpc.core import TanhActivation
from fabricpc.core.activations import SoftmaxActivation
from fabricpc.core.energy import CrossEntropyEnergy
from fabricpc.core.inference import InferenceSGD
from fabricpc.core.initializers import XavierInitializer
from fabricpc.training import train_pcn, evaluate_pcn
from fabricpc.experiments import ExperimentArm, PlannedMultiContrastExperiment
from fabricpc.utils.data.dataloader import (
    FashionMnistLoader,
    FewShotLoader,
    NoisyTestLoader,
)
from fabricpc import setup_jax

setup_jax()
jax.config.update("jax_default_prng_impl", "threefry2x32")


BATCH_SIZE = 64
HIDDEN_WIDTH = 64

ARCH_CONFIGS: Dict[str, List[str]] = {
    "MLP": ["Linear", "Linear", "Linear"],
    "1hopfield": ["Linear", "StorkeyHopfield", "Linear"],
    "1hopfield-late": ["Linear", "Linear", "StorkeyHopfield"],
    "2hopfield": ["Linear", "StorkeyHopfield", "StorkeyHopfield"],
}

## Model Factory

In [2]:
def make_model_factory(layer_types: Sequence[str], hopfield_strength, width: int = HIDDEN_WIDTH):
    """Return a model_factory(rng_key) for a classifier with the given hidden stack."""
    if not layer_types:
        raise ValueError("layer_types must contain at least one layer.")
    if layer_types[0] != "Linear":
        raise ValueError(
            f"Position 0 of layer_types must be 'Linear' (the 784 -> {width} "
            f"feature-extraction projection); got {layer_types[0]!r}. "
            "StorkeyHopfield is width-preserving and cannot change dimension."
        )

    def create_model(rng_key):
        pixels = IdentityNode(shape=(784,), name="pixels")
        nodes: List[NodeBase] = [pixels]
        edges: List[Edge] = []
        prev: NodeBase = pixels

        for i, ltype in enumerate(layer_types):
            if ltype == "StorkeyHopfield":
                layer = StorkeyHopfield(
                    shape=(width,),
                    name=f"hidden{i}",
                    hopfield_strength=hopfield_strength,
                )
            elif ltype == "Linear":
                layer = Linear(
                    shape=(width,),
                    activation=TanhActivation(),
                    name=f"hidden{i}",
                    weight_init=XavierInitializer(),
                )
            else:
                raise ValueError(
                    f"Unknown layer type {ltype!r} at position {i}; "
                    "expected 'Linear' or 'StorkeyHopfield'."
                )
            nodes.append(layer)
            edges.append(Edge(source=prev, target=layer.slot("in")))
            prev = layer

        output = Linear(
            shape=(10,),
            activation=SoftmaxActivation(),
            energy=CrossEntropyEnergy(),
            name="class",
            weight_init=XavierInitializer(),
        )
        nodes.append(output)
        edges.append(Edge(source=prev, target=output.slot("in")))

        structure = graph(
            nodes=nodes,
            edges=edges,
            task_map=TaskMap(x=pixels, y=output),
            inference=InferenceSGD(eta_infer=0.05, infer_steps=20),
        )
        params = initialize_params(structure, rng_key)
        return params, structure

    return create_model


def arch_str(layer_types: Sequence[str], width: int = HIDDEN_WIDTH) -> str:
    """Human-readable architecture string for the header print."""
    parts = ["784"]
    for ltype in layer_types:
        node = "StorkeyHopfield" if ltype == "StorkeyHopfield" else "Linear"
        parts.append(f"{width}({node}, tanh)")
    parts.append("10(softmax, CE)")
    return " -> ".join(parts)

## Data Factory

In [3]:
def make_data_factory(k_per_class, noise_std, batch_size):
    """Return a data_loader_factory(seed) for the experiment runner."""
    noise_level_id = int(round(noise_std * 1_000_000))

    def factory(seed):
        train_seed = int(np.random.SeedSequence([seed, 0]).generate_state(1)[0])
        noise_seed = int(
            np.random.SeedSequence([seed, 1, noise_level_id]).generate_state(1)[0]
        )

        train_loader = FewShotLoader(
            dataset_name="fashion_mnist",
            split="train",
            k_per_class=k_per_class,
            batch_size=batch_size,
            num_classes=10,
            shuffle=True,
            seed=train_seed,
            tensor_format="flat",
            normalize_mean=0.2860,
            normalize_std=0.3530,
        )

        base_test_loader = FashionMnistLoader(
            split="test",
            batch_size=batch_size,
            shuffle=False,
            tensor_format="flat",
        )

        test_loader = NoisyTestLoader(
            base_loader=base_test_loader,
            noise_std=noise_std,
            seed=noise_seed,
        )

        return train_loader, test_loader

    return factory

## Experiment Helpers

In [4]:
def build_contrasts(hopfield_arms_to_run: List[str]) -> List[Tuple[str, str]]:
    """Build the planned-contrast list from the selected Hopfield arms."""
    has_1hop = "1hopfield" in hopfield_arms_to_run
    has_late = "1hopfield-late" in hopfield_arms_to_run
    has_2hop = "2hopfield" in hopfield_arms_to_run

    contrasts: List[Tuple[str, str]] = []
    if has_1hop:
        contrasts.append(("1hopfield", "MLP"))
        if has_2hop:
            contrasts.append(("2hopfield", "1hopfield"))
        if has_late:
            contrasts.append(("1hopfield-late", "1hopfield"))
    elif has_late:
        contrasts.append(("1hopfield-late", "MLP"))
        if has_2hop:
            contrasts.append(("2hopfield", "1hopfield-late"))
    elif has_2hop:
        contrasts.append(("2hopfield", "MLP"))
    return contrasts


def _pct(x: float) -> str:
    return f"{x * 100:+.2f}"


def print_per_arm_table(arm_names: List[str], grid: List[Dict]):
    print()
    print("Per-arm accuracy (mean +/- SE, %):")
    cols = [f"{'K':>5}", f"{'Noise':>6}"]
    for name in arm_names:
        cols.append(f"{name:>14}")
    header = "  ".join(cols)
    print(header)
    print("-" * len(header))
    for r in grid:
        parts = [f"{r['k']:>5}", f"{r['noise']:>6.1f}"]
        for name in arm_names:
            cell = f"{r[f'{name}_mean'] * 100:6.2f}+/-" f"{r[f'{name}_se'] * 100:.2f}"
            parts.append(f"{cell:>14}")
        print("  ".join(parts))


def print_contrasts_table(contrasts: List[Tuple[str, str]], grid: List[Dict], n_trials: int):
    if not contrasts:
        return
    print()
    print("Planned contrasts (paired two-sided t-test on per-trial differences):")
    delta_widths = [max(len(f"{a}-{b} D%"), 16) for a, b in contrasts]
    cols = [f"{'K':>5}", f"{'Noise':>6}"]
    for (a, b), w in zip(contrasts, delta_widths):
        label = f"{a}-{b}"
        cols.append(f"{label + ' D%':>{w}}")
        cols.append(f"{'p':>8}")
        cols.append(f"{'sig':>4}")
        cols.append(f"{'d':>7}")
    header = "  ".join(cols)
    print(header)
    print("-" * len(header))
    for r in grid:
        parts = [f"{r['k']:>5}", f"{r['noise']:>6.1f}"]
        for (a, b), w in zip(contrasts, delta_widths):
            key = f"{a}-{b}"
            delta_str = _pct(r[f"{key}_delta"])
            parts.append(f"{delta_str:>{w}}")
            p = r[f"{key}_pval"]
            parts.append(f"{p:8.4f}" if not np.isnan(p) else f"{'n/a':>8}")
            parts.append(f"{'*' if r[f'{key}_sig'] else '':>4}")
            d = r[f"{key}_d"]
            parts.append(f"{d:7.3f}" if not np.isnan(d) else f"{'n/a':>7}")
        print("  ".join(parts))
    print()


def print_descriptive_total_delta(grid: List[Dict]):
    if not grid or "2hopfield-MLP_delta_descriptive" not in grid[0]:
        return
    print()
    print("Reported-only (descriptive, NOT a planned contrast):")
    print(f"{'K':>5}  {'Noise':>6}  {'2hopfield-MLP D%':>18}  {'SE':>8}")
    print("-" * 44)
    for r in grid:
        d = r["2hopfield-MLP_delta_descriptive"]
        se = r["2hopfield-MLP_se_descriptive"]
        print(f"{r['k']:>5}  {r['noise']:>6.1f}  " f"{_pct(d):>18}  {se * 100:>7.2f}")


def _legend_text(n_trials: int, n_cells: int, contrasts: List[Tuple[str, str]]) -> str:
    n_contrasts = len(contrasts)
    total_tests = n_cells * n_contrasts
    cumulative_rule = ""
    if ("1hopfield", "MLP") in contrasts and ("2hopfield", "1hopfield") in contrasts:
        cumulative_rule = (
            "  Per-cell criterion for 'cumulative gain': BOTH increment contrasts\n"
            "  (1hopfield-MLP and 2hopfield-1hopfield) starred with positive\n"
            "  deltas (intersection-union test of the conjunction, valid at\n"
            "  level alpha under arbitrary dependence).\n"
        )
    return (
        f"Legend:\n"
        f"  * = two-sided paired t-test p<0.05 on the per-trial difference\n"
        f"      vector, uncorrected across {n_cells} cells x {n_contrasts} "
        f"contrasts = {total_tests} tests.\n"
        f"      Per-cell stars are exploratory across the grid.\n"
        f"{cumulative_rule}"
        f"  n_trials = {n_trials}; Cohen's d is the paired d on the same vector."
    )


def print_heatmaps(contrasts, k_values, noise_levels, grid):
    if len(k_values) <= 1 or len(noise_levels) <= 1:
        return
    for a, b in contrasts:
        print()
        print(f"Delta Accuracy Heatmap ({a} - {b}, percentage points):")
        print()
        header = f"{'K':>5}  " + "  ".join(f"n={n:.1f}" for n in noise_levels)
        print(header)
        print("-" * len(header))
        for k in k_values:
            row = [f"{k:>5}"]
            for n in noise_levels:
                match = next((r for r in grid if r["k"] == k and r["noise"] == n), None)
                if match:
                    key = f"{a}-{b}"
                    delta = match[f"{key}_delta"] * 100
                    sig = match[f"{key}_sig"]
                    row.append(f"{delta:+6.1f}{'*' if sig else ' '}")
                else:
                    row.append(f"{'n/a':>7}")
            print("  ".join(row))

## Configuration

In [5]:
# Configuration (replaces argparse)
k_values = [500]        # K shots per class values (list of ints)
noise_levels = [2.0]    # Noise std values (list of floats)
n_trials = 5            # Number of paired trials per condition
num_epochs = 1          # Training epochs per trial
strength = 2.0          # Hopfield strength (float or None for learnable)
networks = ["1hopfield", "1hopfield-late", "2hopfield"]  # Hopfield variants to test
verbose = False

## Run Experiment

In [6]:
hopfield_strength = strength if isinstance(strength, float) else None

print("=" * 70)
print("StorkeyHopfield vs MLP \u2014 Few-Shot + Noise-Robustness Demo")
print("=" * 70)
print(f"Architecture: all arms use depth-4 graph with hidden_width={HIDDEN_WIDTH}")
print(f"Dataset: Fashion-MNIST")
print(f"k_values: {k_values}, noise_levels: {noise_levels}")
print(f"n_trials: {n_trials}, num_epochs: {num_epochs}, hopfield_strength: {hopfield_strength}")
print(f"Networks: MLP + {networks}")
print()

# Build arms
all_arm_names = ["MLP"] + networks
contrasts = build_contrasts(networks)

arms = {}
for name in all_arm_names:
    layer_types = ARCH_CONFIGS[name]
    arms[name] = ExperimentArm(
        name=name,
        model_factory=make_model_factory(layer_types, hopfield_strength),
        train_fn=train_pcn,
        eval_fn=evaluate_pcn,
        optimizer=optax.adamw(0.001, weight_decay=0.001),
        train_config={"num_epochs": num_epochs},
    )

# Run grid
grid = []
for k in k_values:
    for noise in noise_levels:
        print(f"\n--- K={k}, noise_std={noise} ---")
        data_factory = make_data_factory(k, noise, BATCH_SIZE)

        experiment = PlannedMultiContrastExperiment(
            arms=arms,
            contrasts=contrasts,
            metric="accuracy",
            data_loader_factory=data_factory,
            n_trials=n_trials,
            verbose=verbose,
        )

        cell_results = experiment.run().to_dict()
        cell_results["k"] = k
        cell_results["noise"] = noise
        grid.append(cell_results)

# Print results
print("\n" + "=" * 70)
print_per_arm_table(all_arm_names, grid)
print_contrasts_table(contrasts, grid, n_trials)
print_descriptive_total_delta(grid)
print_heatmaps(contrasts, k_values, noise_levels, grid)

StorkeyHopfield vs MLP — Few-Shot + Noise-Robustness Demo
Architecture: all arms use depth-4 graph with hidden_width=64
Dataset: Fashion-MNIST
k_values: [500], noise_levels: [2.0]
n_trials: 5, num_epochs: 1, hopfield_strength: 2.0
Networks: MLP + ['1hopfield', '1hopfield-late', '2hopfield']


--- K=500, noise_std=2.0 ---
--- Trial 1/5 (seed=0) ---


  0%|          | 0/79 [00:00<?, ?it/s]

  MLP: accuracy=0.5829  (train: 17.5s)


  0%|          | 0/79 [00:00<?, ?it/s]

  1hopfield: accuracy=0.6249  (train: 15.6s)


...

  1hopfield: accuracy=0.6380  (train: 16.7s)


  0%|          | 0/79 [00:00<?, ?it/s]

  1hopfield-late: accuracy=0.6244  (train: 18.8s)


  0%|          | 0/79 [00:00<?, ?it/s]

  2hopfield: accuracy=0.6443  (train: 18.6s)


Per-arm accuracy (mean +/- SE, %):
    K   Noise             MLP       1hopfield  1hopfield-late       2hopfield
-----------------------------------------------------------------------------
  500     2.0    58.17+/-0.73    62.76+/-0.42    61.61+/-0.55    64.21+/-0.33

Planned contrasts (paired two-sided t-test on per-trial differences):
    K   Noise  1hopfield-MLP D%         p   sig        d  2hopfield-1hopfield D%         p   sig        d  1hopfield-late-1hopfield D%         p   sig        d
---------------------------------------------------------------------------------------------------------------------------------------------------------------
  500     2.0             +4.59    0.0004     *    4.753                   +1.45    0.0068     *    2.294                        -1.15    0.0530         -1.217


Reported-only (descriptive, NOT a planned contrast):
    K   Noise    2hopfield-MLP D%        SE
----------------------------------